# Spice Wizard — Serve Qwen on the AMD MI300X + public tunnel

Run this notebook **top to bottom** in the AMD cloud Jupyter, then paste the 3
`LLM_*` values it prints into `Spice_Wizard/.env` on your workstation. This
turns the manual copy-paste flow into a **live** connection: the GUI chat and
`generate_verify.py` call the MI300X directly.

**Keep this kernel alive** — if it restarts, the model, server, and tunnel die
(and the tunnel URL changes on every restart).

```
Workstation (GUI + LTspice verifier) ──HTTPS/443──▶ cloudflared tunnel ──▶ FastAPI :8000 ──▶ Qwen on MI300X (ROCm)
```

**Why this defeats the "firewall" problem:** nothing inbound is ever opened.
The MI300X box makes an *outbound* connection to Cloudflare pinned to HTTP/2
over TCP 443 — the exact same path `pip install` and the Hugging Face model
download already use successfully from this notebook. Your workstation then
talks to a public `https://…trycloudflare.com` URL, never to the box itself.
(Gradio's share tunnel uses a nonstandard outbound port, which is why it kept
failing here.)

In [ ]:
import os
!pip -q install -U huggingface_hub transformers accelerate safetensors fastapi uvicorn requests

# Fast downloads only if the wrapper is available -- plain HTTP is fine otherwise.
try:
    import hf_transfer  # noqa: F401
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    print("hf_transfer enabled (fast downloads)")
except ImportError:
    os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
    print("hf_transfer not available -- using standard download (fine)")

## 1 · Load the model onto the MI300X

Default is **Qwen2.5-Coder-7B-Instruct** — the best speed/quality point for a
live demo. Swap `MODEL_NAME` to `Qwen/Qwen2.5-Coder-32B-Instruct` (the 192 GB
card handles it easily, just slower per token) or `...-1.5B-Instruct`
(fastest) and re-run this cell. Weights persist in `~/models/`, so re-runs
skip the download.

In [ ]:
import glob, os, torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-Coder-7B-Instruct")
local_dir = os.path.expanduser(f"~/models/{MODEL_NAME.split('/')[-1].lower()}")  # persists (NOT /tmp)

if not glob.glob(os.path.join(local_dir, "*.safetensors")):
    print("Downloading (first time only)...")
    snapshot_download(repo_id=MODEL_NAME, local_dir=local_dir)
else:
    print("Model already on disk -- skipping download.")

tok = AutoTokenizer.from_pretrained(local_dir)
model = AutoModelForCausalLM.from_pretrained(
    local_dir, dtype=torch.float16, device_map={"": 0}   # whole model on the MI300X
)
model_name = MODEL_NAME
print("Loaded on:", next(model.parameters()).device)  # ROCm maps cuda:0 -> the MI300X

## 2 · Prove inference runs on the MI300X

**Open a Terminal and run `watch -n 0.5 rocm-smi` while the next cell runs** —
GPU% / VRAM% jumping is your AMD-compute evidence. Screenshot it for the
README and the demo video (see `docs/AMD_DEPLOYMENT.md`, section 6).

In [ ]:
messages = [{"role": "user",
             "content": "In this LTspice line, change the gain from 2 to 5 by editing only R2: "
                        "'R2 OUT N001 1K'. Reply with only the new line."}]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

## 3 · OpenAI-compatible server (reuses the loaded model — no reload)

Serves `/v1/chat/completions` on port 8000 in a background thread inside this
kernel. One request at a time (a lock guards `generate`) — exactly right for
the sequential generate→verify→retry loop; for best-of-N, request candidates
one after another. Requests must send `Authorization: Bearer <API_TOKEN>`;
everything else gets a 401, which matters because the tunnel URL is public
internet.

The server ignores the `model` field in requests and always uses the loaded
model, so the `LLM_MODEL` value on the workstation is cosmetic (it just shows
up in logs).

In [ ]:
import secrets, threading, time, uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

API_TOKEN = os.environ.get("SPICE_WIZARD_API_TOKEN") or secrets.token_urlsafe(16)
print("API token for this session:", API_TOKEN)

app = FastAPI()
_gen_lock = threading.Lock()          # transformers generate() is not thread-safe

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": model_name, "object": "model"}]}

@app.post("/v1/chat/completions")
async def chat(request: Request):
    if request.headers.get("authorization", "") != f"Bearer {API_TOKEN}":
        return JSONResponse({"error": "bad or missing bearer token"}, status_code=401)
    body = await request.json()
    msgs = body.get("messages", [])
    max_new = min(int(body.get("max_tokens") or 1024), 4096)
    temp = float(body.get("temperature") or 0)

    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=max_new)
    kw.update(dict(do_sample=True, temperature=temp) if temp > 0 else dict(do_sample=False))
    with _gen_lock:
        gen = model.generate(**inputs, **kw)

    completion = tok.decode(gen[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    n_prompt, n_total = int(inputs.input_ids.shape[1]), int(gen.shape[1])
    return {
        "id": "chatcmpl-mi300x", "object": "chat.completion", "model": model_name,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": completion},
                     "finish_reason": "stop"}],
        "usage": {"prompt_tokens": n_prompt, "completion_tokens": n_total - n_prompt,
                  "total_tokens": n_total},
    }

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print("server up on :8000")

In [ ]:
# Confirm the server works locally BEFORE tunneling
# (isolates 'server broken' from 'network broken').
import requests
r = requests.post("http://localhost:8000/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_TOKEN}"},
    json={"messages": [{"role": "user",
                        "content": "Change R2 to give gain 5: 'R2 OUT N001 1K'. Reply with only the new line."}],
          "max_tokens": 64, "temperature": 0})
print(r.status_code)
print(r.json()["choices"][0]["message"]["content"])

## 4 · Public URL via cloudflared — pinned to HTTPS/443

Downloads the `cloudflared` binary with Python (no curl needed) and starts a
**quick tunnel**: no account, no signup, no inbound firewall rule.

`--protocol http2` forces the tunnel's outbound connection onto **TCP 443** —
the same port your pip installs and model downloads already use — instead of
cloudflared's default QUIC on UDP 7844, which restrictive egress firewalls
(like this cloud's) commonly block.

⚠️ The URL is different every time this cell (re)runs — re-paste into `.env`
after any restart.

In [ ]:
import os, re, stat, subprocess, threading, time, urllib.request

BIN = os.path.expanduser("~/cloudflared")
if not os.path.exists(BIN):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        BIN)
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC)
    print("cloudflared ready")

try:
    _tunnel.terminate()   # kill a previous tunnel if this cell is re-run
    time.sleep(1)
except NameError:
    pass

_tunnel = subprocess.Popen(
    [BIN, "tunnel", "--url", "http://localhost:8000",
     "--protocol", "http2",          # outbound over TCP 443 -- firewall-friendly
     "--edge-ip-version", "auto",
     "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url, lines = None, []
url_re = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
deadline = time.time() + 90
while time.time() < deadline and public_url is None:
    line = _tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    lines.append(line.rstrip())
    m = url_re.search(line)
    if m:
        public_url = m.group(0)

# keep draining tunnel output so the process never blocks on a full pipe
threading.Thread(target=lambda: [None for _ in _tunnel.stdout], daemon=True).start()

if public_url:
    print("PUBLIC URL:", public_url)
    print("\n=== paste into Spice_Wizard/.env on your workstation ===")
    print(f"LLM_BASE_URL={public_url}/v1")
    print(f"LLM_API_KEY={API_TOKEN}")
    print(f"LLM_MODEL={model_name}")
else:
    print("No URL after 90s -- cloudflared output so far:")
    print("\n".join(lines[-30:]))

In [ ]:
# End-to-end self test THROUGH the public URL -- if this passes, your workstation will work.
import requests
r = requests.post(f"{public_url}/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_TOKEN}"},
    json={"messages": [{"role": "user", "content": "Say READY if you can hear me."}],
          "max_tokens": 8, "temperature": 0}, timeout=120)
print(r.status_code, r.json()["choices"][0]["message"]["content"])
print("\nTunnel verified end-to-end -- the workstation can now reach the MI300X.")

## 5 · On your workstation — set 3 env values, zero code changes

In `Spice_Wizard/.env` (create it from `.env.example` if needed), set the
values printed by the tunnel cell:

```
LLM_BASE_URL=https://<printed-above>.trycloudflare.com/v1
LLM_API_KEY=<printed-above>
LLM_MODEL=Qwen/Qwen2.5-Coder-7B-Instruct
```

Test the chain (should print a reply generated on the MI300X):

```bash
cd Spice_Wizard
python -c "from llm_client import call_llm; print(call_llm('say hi in 3 words'))"
```

Run the live verify loop — **keep `rocm-smi` visible on the box while it
runs**; GPU% spiking on a request your workstation triggered is the
demo-video money shot:

```bash
python generate_verify.py AD8092 \
  --spec "non-inverting gain of 5 V/V, +/-5 V supplies, 100 mV at 1 MHz" \
  --metric gain_db=13.98:1.0 \
  --freq 1e6
```

The GUI chat agent uses the same `LLM_*` variables, so it now talks to the
MI300X too:

```bash
python app/run_gui.py
```

If the tunnel is unavailable during judging, the manual best-of-N flow in
[`docs/AMD_DEPLOYMENT.md`](../docs/AMD_DEPLOYMENT.md) remains a complete
fallback (`--prompt-only` → generate on the MI300X → `--candidates` → Arena).

### Troubleshooting
- **Kernel restarted?** Re-run all cells; the model download is skipped, and a
  **new** tunnel URL + token are printed — update `.env` again.
- **401 from the server** → `LLM_API_KEY` on the workstation doesn't match the
  token printed by the server cell.
- **No URL after 90 s** → re-run the tunnel cell once (Cloudflare quick-tunnel
  capacity hiccups happen); if it still fails, the egress filter is stricter
  than TCP 443 — use the manual flow above for the demo.
- **Slow generations** → you're on 32B; swap to 7B in the model cell for the
  live demo.